# 准备工作
## 数据概览
- 图片数据转换为(3,128,128): 使用PIL 打开图片 读出来 然后使用torchvison下的 Transforms 转化为指定形状的长量
## 复现保证


In [5]:
### 数据操作
import os
base_dir = './food11'
training_files = os.listdir(os.path.join(base_dir,'training'))
test_files = os.listdir(os.path.join(base_dir,'test'))
validation_files = os.listdir(os.path.join(base_dir,'validation'))
### 数据量统计
print(f"训练集合数据 {len(training_files)}")
print(f"测试集合数据 {len(test_files)}")
print(f"验证集合数据 {len(validation_files)}")

训练集合数据 9866
测试集合数据 3347
验证集合数据 3430


In [6]:
### 保证复现
import torch
import numpy as np
seed = 1000
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [11]:
### transformer
import torchvision.transforms as transforms
test_tfm = transforms.Compose([
    transforms.Resize([128,128]),
    transforms.ToTensor()
])

training_tfm = transforms.Compose([
    transforms.Resize([128,128]),
    transforms.ToTensor()
])

# Dataset复写

In [18]:
### 复写Dataset类
from torch.utils.data import Dataset
from PIL import Image
class FoodDataset(Dataset):
    def __init__(self, path, tfm = test_tfm, files = None):
        super().__init__() ### 父类中没有初始化方法
        if files is not None:
            self.files = files
        else:
            self.files = sorted([os.path.join(path,x) for x in os.listdir(path) if x.endswith('.jpg')])
        self.tfm = test_tfm
        print(f"{self.files[0]}")
    def __getitem__(self, idx):
        item = self.files[idx]
        fig = Image.open(item)
        fig = self.tfm(fig)
        try:
            label = int(item.split('/')[-1].split('_')[0])
        except:
            label = -1
        return fig,label
    def __len__(self):
        return len(self.files)

In [20]:
train_data = FoodDataset(os.path.join(base_dir,'training'),training_tfm)
train_data[0][0].shape

./food11/training/0_0.jpg


torch.Size([3, 128, 128])

# 模型
## 卷积部分：
- 输入维度：(batch_size, c, h, w) 此处是 (b,3,128,128)
- 输入卷积层conv2d(3, 64, kernel_size = 3, padding = 1, stride = 1); 输出维度 (b, 64, 128, 128)

    $$ h_{putput} (w_{putput}) = \frac{h_{input} (w_{input}) - kernel\_size + 2 * padding}{stride} + 1  $$
    - ReLU()
    - BN()
- 输入汇聚层maxpooling(kernel_size = 2, stride = 2, padding = 0); 输出维度 (b, 64, 64, 64)
- 输入卷积层conv2d(64, 128, kernel_size = 3, padding = 1, stride = 1); 输出维度 (b, 128, 64, 64)
    - ReLU()
    - BN()
- 输入汇聚层maxpooling(kernel_size =2, stride = 2, padding = 0); 输出维度 (b, 128, 32, 32)
- 输入卷积层conv2d(128, 256, kernel_size = 3, stride = 1, padding = 1); 输出维度 (b, 256, 32, 32)
    - ReLU()
    - BN()
- 输入汇聚层maxpooling(kernel_size =2, stride = 2, padding = 0); 输出维度 (b, 256, 16, 16)
- 输入卷积层conv2d(256, 512, kernel_size = 3, stride = 1, padding = 1); 输出维度 (b, 512, 16, 16)
    - ReLU()
    - BN()
- 输入汇聚层maxpooling(kernel_size =2, stride = 2, padding = 0); 输出维度 (b, 512, 8, 8)
- 输入卷积层conv2d(512, 512, kernel_size = 3, stride = 1, padding = 1); 输出维度 (b, 512, 8, 8)
    - ReLU()
    - BN()
- 输入汇聚层maxpooling(kernel_size =2, stride = 2, padding = 0); 输出维度 (b, 512, 4, 4)
## FC
— 输入二维张量：(1) 使用view(-1, 512*4*4) 转化为（b, 512*4*4 (2) 使用flatten(1,-1)进行转化
- 输入nn.Linear(512*4*4, 1024); 输出  (b, 1024)
    - ReLU()
- 输入nn.Linear(1024, 512); 输出 (b, 512)
    - ReLU()
- 任务层 输入nn.Liear(512, 11); 输出 (b, 11) ======> logits

In [ ]:
### model
### 我这里写两层
import torch.nn as nn
class Conv(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),  # [64, 128, 128] ### (64, 128, 128)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [64, 64, 64] ### (64, 64, 64)

            nn.Conv2d(64, 128, 3, 1, 1), # [128, 64, 64] ### (128, 64, 64)
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [128, 32, 32]. ### (128, 32, 32)

            nn.Conv2d(128, 256, 3, 1, 1), # [256, 32, 32] ### (256, 32, 32)
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [256, 16, 16]

            nn.Conv2d(256, 512, 3, 1, 1), # [512, 16, 16]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),       # [512, 8, 8]
            
            nn.Conv2d(512, 512, 3, 1, 1), # [512, 8, 8]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),       # [512, 4, 4]
        )
    def forward(self, x):
        return self.conv_layers(x)

# 产生DataLoader

# 定义Loss，optimizer